# encoder-decoder-symmetric composite — cx8: symmetric AE as an nn.Module subclass (encoder + decoder children)

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `encoder-decoder-symmetric`, `nn-module-subclass`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
from einops.layers.torch import Rearrange

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "encoder-decoder-symmetric"
DD_ATOM_IDS = ["encoder-decoder-symmetric", "nn-module-subclass"]
DD_SUBTOPICS = ["CNN: Encoder-decoder symmetric layout", "PyTorch: nn.Module subclassing"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

An autoencoder is the canonical example of why `nn.Module` subclassing is needed: the forward pass is *not* just `seq(x)`. It's a SEQUENCE of stages that you may also want to inspect individually (e.g. visualizing the encoded features). The right pattern is:

- Hold the **encoder** as one child Module (a `Sequential`, say).
- Hold the **decoder** as another child Module.
- Subclass `nn.Module`, register both children in `__init__` (after `super().__init__()`), and write `forward(self, x): return self.decoder(self.encoder(x))`.

**Why both atoms together.** The symmetric layout is the architecture — it determines the shape contract `model(x).shape == x.shape`. The module-subclass wrapping is the API — it gives you `.parameters()`, `.to(device)`, `.train()/.eval()`, and a place to expose an `.encode(x)` helper. Either alone is incomplete.

**Anatomy.**
```python
class SymAE(nn.Module):
    def __init__(self, c):
        super().__init__()                       # nn-module-subclass: required first call.
        self.encoder = nn.Sequential(            # encoder-decoder-symmetric: down stages.
            nn.Conv2d(c, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.decoder = nn.Sequential(            # encoder-decoder-symmetric: up stages.
            nn.Upsample(scale_factor=2), nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2), nn.Conv2d(16, c, 3, padding=1),
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))
```

### Composite Exercise — symmetric AE as an nn.Module subclass (encoder + decoder children)

**Atoms exercised together**: `encoder-decoder-symmetric`, `nn-module-subclass`

Implement `cx8_make_sym_ae_class()` — return the CLASS `SymAE` (not an instance).

Required structure (`class SymAE(nn.Module)`):
1. `__init__(self, in_channels)`:
   - `super().__init__()` first.
   - `self.encoder = nn.Sequential(`
         `nn.Conv2d(in_channels, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),`
         `nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),`
     `)`
   - `self.decoder = nn.Sequential(`
         `nn.Upsample(scale_factor=2), nn.Conv2d(32, 16, kernel_size=3, padding=1), nn.ReLU(),`
         `nn.Upsample(scale_factor=2), nn.Conv2d(16, in_channels, kernel_size=3, padding=1),`
     `)`
2. `forward(self, x): return self.decoder(self.encoder(x))`.

Test checks:
- `cx8_make_sym_ae_class()` returns a CLASS (not an instance) that subclasses `nn.Module`.
- An instance has BOTH `encoder` and `decoder` registered as named children.
- `list(model.parameters())` is non-empty (proves `super().__init__()` was called).
- `model(x).shape == x.shape` for several input sizes (spatial dim divisible by 4).

In [ ]:
def cx8_make_sym_ae_class():
    class SymAE(nn.Module):
        def __init__(self, in_channels):
            # Atom B (nn-module-subclass): super() FIRST — wires registry.
            super().__init__()
            # Atom A (encoder-decoder-symmetric): mirrored down/up stages.
            self.encoder = nn.Sequential(
                nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(16, 32, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
            )
            self.decoder = nn.Sequential(
                nn.Upsample(scale_factor=2),
                nn.Conv2d(32, 16, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.Upsample(scale_factor=2),
                nn.Conv2d(16, in_channels, kernel_size=3, padding=1),
            )

        def forward(self, x):
            return self.decoder(self.encoder(x))

    return SymAE


<details><summary>Show solution — cx8</summary>

```python
def cx8_make_sym_ae_class():
    class SymAE(nn.Module):
        def __init__(self, in_channels):
            # Atom B (nn-module-subclass): super() FIRST — wires registry.
            super().__init__()
            # Atom A (encoder-decoder-symmetric): mirrored down/up stages.
            self.encoder = nn.Sequential(
                nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(16, 32, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
            )
            self.decoder = nn.Sequential(
                nn.Upsample(scale_factor=2),
                nn.Conv2d(32, 16, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.Upsample(scale_factor=2),
                nn.Conv2d(16, in_channels, kernel_size=3, padding=1),
            )

        def forward(self, x):
            return self.decoder(self.encoder(x))

    return SymAE
```

Returning the CLASS (not an instance) is the idiom we use when the test needs to construct multiple instances with different channel counts. The encoder+decoder name registration lets you do `model.encoder(x)` to inspect the encoded feature map, which is how reconstruction-quality visualizations are built.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx8'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx8',
        'subtopics': ["CNN: Encoder-decoder symmetric layout", "PyTorch: nn.Module subclassing"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()